In [5]:
import pandas as pd
import numpy as np

# Человекочитаемый вывод чисел в pandas (для отображения в ноутбуке)
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.options.display.float_format = "{:,.2f}".format

# === Параметры ===
PATH = r"C:\Users\Ксения\Downloads\Тестовое_задание_Data_аналитик_ЦО_2025_.xlsx"
SHEET = "Данные для задачи 3"


In [6]:
df = pd.read_excel(PATH, sheet_name=SHEET)

required_cols = {"Бренд AX 1", "Неделя", "Выручка", "Маржинальность"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"На листе '{SHEET}' не хватает колонок: {sorted(missing)}")

df = df.copy()

# типы
df["Неделя"] = pd.to_numeric(df["Неделя"], errors="coerce").astype("Int64")
df["Выручка"] = pd.to_numeric(df["Выручка"], errors="coerce")
df["Маржинальность"] = pd.to_numeric(df["Маржинальность"], errors="coerce")

# чистка
df = df.dropna(subset=["Бренд AX 1", "Неделя", "Выручка", "Маржинальность"])
df = df[df["Выручка"] >= 0]
df = df[(df["Маржинальность"] >= -1) & (df["Маржинальность"] <= 1)]

weeks = sorted(df["Неделя"].unique())
if len(weeks) < 2:
    raise ValueError(f"В данных меньше двух недель: {weeks}")

w_last = int(weeks[-1])   # W-1 (последняя неделя в данных)
w_prev = int(weeks[-2])   # W-2 (предыдущая)

print(f"Сравниваем недели: W-2={w_prev} vs W-1={w_last}")
print("Количество строк в исходном листе:", len(df))


Сравниваем недели: W-2=14 vs W-1=15
Количество строк в исходном листе: 104


In [15]:
# === Ячейка 3 (объединённая: бывшие ячейки 3 + 4) ===
# Агрегация продаж по брендам за W-2 и W-1 + дельты + обработка NaN (новый/пропал бренд) + витрина для вывода

# 1) Маржа в рублях (если в исходнике есть только маржинальность-доля)
df["Маржа_руб"] = df["Выручка"] * df["Маржинальность"]

# 2) Берём только две недели
df2 = df[df["Неделя"].isin([w_prev, w_last])].copy()

# 3) Пивот: выручка и маржа по брендам
pivot = (
    df2.pivot_table(
        index="Бренд AX 1",
        columns="Неделя",
        values=["Выручка", "Маржа_руб"],
        aggfunc="sum"
    )
)

# 4) Плоские названия колонок
pivot.columns = [f"{metric}_W{week}" for metric, week in pivot.columns]
pivot = pivot.reset_index()

# 5) Защита: если какой-то недели нет в pivot — создаём колонку с нулями
for metric in ["Выручка", "Маржа_руб"]:
    for w in [w_prev, w_last]:
        col = f"{metric}_W{w}"
        if col not in pivot.columns:
            pivot[col] = 0.0
        pivot[col] = pivot[col].fillna(0.0)

# 6) Маржинальность по бренду (если выручка=0 -> NaN)
def safe_margin_pct(margin_rub: pd.Series, revenue: pd.Series) -> pd.Series:
    return pd.Series(np.where(revenue > 0, margin_rub / revenue, np.nan), index=revenue.index)

col_mp_prev = f"Маржинальность_W{w_prev}"
col_mp_last = f"Маржинальность_W{w_last}"

pivot[col_mp_prev] = safe_margin_pct(pivot[f"Маржа_руб_W{w_prev}"], pivot[f"Выручка_W{w_prev}"])
pivot[col_mp_last] = safe_margin_pct(pivot[f"Маржа_руб_W{w_last}"], pivot[f"Выручка_W{w_last}"])

# 7) Дельты
pivot["ΔВыручка"] = pivot[f"Выручка_W{w_last}"] - pivot[f"Выручка_W{w_prev}"]
pivot["ΔМаржа_руб"] = pivot[f"Маржа_руб_W{w_last}"] - pivot[f"Маржа_руб_W{w_prev}"]
pivot["ΔМаржинальность_п.п."] = (pivot[col_mp_last] - pivot[col_mp_prev]) * 100

# 8) Обработка NaN: флаги наличия продаж и комментарий
pivot["sold_W2"] = pivot[f"Выручка_W{w_prev}"] > 0
pivot["sold_W1"] = pivot[f"Выручка_W{w_last}"] > 0

pivot["comment"] = np.select(
    [
        (~pivot["sold_W2"]) & (pivot["sold_W1"]),
        (pivot["sold_W2"]) & (~pivot["sold_W1"]),
        (~pivot["sold_W2"]) & (~pivot["sold_W1"]),
    ],
    [
        "Новый бренд в W-1 (в W-2 продаж не было)",
        "Продажи пропали в W-1 (в W-2 были)",
        "Нет продаж в обе недели",
    ],
    default=""
)

# 9) Витрина для отображения: округляем дельту маржинальности (п.п.)
pivot["ΔМаржинальность_п.п._view"] = pivot["ΔМаржинальность_п.п."].round(3)

pivot_view = pivot[[
    "Бренд AX 1",
    f"Выручка_W{w_prev}", f"Выручка_W{w_last}",
    f"Маржа_руб_W{w_prev}", f"Маржа_руб_W{w_last}",
    col_mp_prev, col_mp_last,
    "ΔВыручка", "ΔМаржа_руб", "ΔМаржинальность_п.п._view",
    "comment"
]].copy()

pivot_view.head(30)


,Бренд AX 1,Выручка_W14,Выручка_W15,Маржа_руб_W14,Маржа_руб_W15,Маржинальность_W14,Маржинальность_W15,ΔВыручка,ΔМаржа_руб,ΔМаржинальность_п.п._view,comment
0,Ahava,"44,410,651.58","73,153,235.14","5,523,678.12","5,996,761.88",0.12,0.08,"28,742,583.56","473,083.76",-4.24,
1,Anne Semonin,"32,673,568.40","19,243,952.97","6,604,613.48","3,878,819.20",0.20,0.20,"-13,429,615.43","-2,725,794.28",-0.06,
2,BABOR,"309,282,334.73","290,841,977.18","45,687,429.72","43,033,598.17",0.15,0.15,"-18,440,357.55","-2,653,831.55",0.02,
3,Bellefontaine,"17,834,942.22",0.00,"1,919,656.22",0.00,0.11,NaN,"-17,834,942.22","-1,919,656.22",NaN,Продажи пропали в W-1 (в W-2 были)
4,Biothal,0.00,"17,949,429.01",0.00,"2,542,770.51",NaN,0.14,"17,949,429.01","2,542,770.51",NaN,Новый бренд в W-1 (в W-2 продаж не было)
5,Biotherm,"223,614,213.98","217,137,765.03","26,067,808.38","25,223,299.35",0.12,0.12,"-6,476,448.95","-844,509.03",-0.04,
6,Botany,"31,572,158.27","28,789,604.70","7,566,499.40","6,979,303.18",0.24,0.24,"-2,782,553.57","-587,196.22",0.28,
7,CHANEL,"236,703,280.29","217,319,320.17","20,775,902.84","16,796,437.58",0.09,0.08,"-19,383,960.12","-3,979,465.25",-1.05,
8,CHOLLEY,0.00,"16,931,694.37",0.00,"1,933,309.45",NaN,0.11,"16,931,694.37","1,933,309.45",NaN,Новый бренд в W-1 (в W-2 продаж не было)
9,COOLA,"22,413,754.85","30,631,097.05","4,639,000.84","6,269,777.87",0.21,0.20,"8,217,342.20","1,630,777.04",-0.23,


In [16]:
company_prev_rev = float(pivot[f"Выручка_W{w_prev}"].sum())
company_last_rev = float(pivot[f"Выручка_W{w_last}"].sum())

company_prev_margin = float(pivot[f"Маржа_руб_W{w_prev}"].sum())
company_last_margin = float(pivot[f"Маржа_руб_W{w_last}"].sum())

company_prev_mp = company_prev_margin / company_prev_rev if company_prev_rev > 0 else np.nan
company_last_mp = company_last_margin / company_last_rev if company_last_rev > 0 else np.nan

company_summary = pd.DataFrame([
    {"Неделя": w_prev, "Выручка": company_prev_rev, "Маржа_руб": company_prev_margin, "Маржинальность": company_prev_mp},
    {"Неделя": w_last, "Выручка": company_last_rev, "Маржа_руб": company_last_margin, "Маржинальность": company_last_mp},
])
company_summary["Маржинальность_%"] = company_summary["Маржинальность"] * 100
company_summary["Выручка"] = company_summary["Выручка"].round(0)
company_summary["Маржа_руб"] = company_summary["Маржа_руб"].round(0)

company_summary


,Неделя,Выручка,Маржа_руб,Маржинальность,Маржинальность_%
0,14,"8,464,188,705.00","1,237,987,713.00",0.15,14.63
1,15,"8,207,907,632.00","1,268,186,314.00",0.15,15.45


In [17]:
# доли выручки
pivot["share_prev"] = np.where(company_prev_rev > 0, pivot[f"Выручка_W{w_prev}"] / company_prev_rev, 0.0)
pivot["share_last"] = np.where(company_last_rev > 0, pivot[f"Выручка_W{w_last}"] / company_last_rev, 0.0)

# маржинальности брендов (NaN -> 0 только для расчёта эффектов)
m_prev = pivot[col_mp_prev].fillna(0.0)
m_last = pivot[col_mp_last].fillna(0.0)

s_prev = pivot["share_prev"]
s_last = pivot["share_last"]

# симметричное разложение (точно сходится)
pivot["effect_margin"] = ((s_prev + s_last) / 2) * (m_last - m_prev)
pivot["effect_mix"] = ((m_prev + m_last) / 2) * (s_last - s_prev)
pivot["effect_total"] = pivot["effect_margin"] + pivot["effect_mix"]

effects_summary = pd.DataFrame([{
    "ΔМаржинальность_компании_п.п.": (company_last_mp - company_prev_mp) * 100,
    "Эффект_маржи_п.п.": pivot["effect_margin"].sum() * 100,
    "Эффект_микса_п.п.": pivot["effect_mix"].sum() * 100,
    "Контроль_сумма_п.п.": pivot["effect_total"].sum() * 100,
}]).round(3)

effects_summary


,ΔМаржинальность_компании_п.п.,Эффект_маржи_п.п.,Эффект_микса_п.п.,Контроль_сумма_п.п.
0,0.82,0.54,0.28,0.82


In [18]:
drivers = pivot[[
    "Бренд AX 1", "ΔВыручка", "ΔМаржа_руб", "ΔМаржинальность_п.п.",
    "effect_total", "effect_margin", "effect_mix"
]].copy()

drivers["effect_total_п.п."] = (drivers["effect_total"] * 100).round(3)
drivers["effect_margin_п.п."] = (drivers["effect_margin"] * 100).round(3)
drivers["effect_mix_п.п."] = (drivers["effect_mix"] * 100).round(3)

top_neg = drivers.sort_values("effect_total_п.п.").head(10)
top_pos = drivers.sort_values("effect_total_п.п.", ascending=False).head(10)

avg_last = company_last_mp
quick = pivot.copy()
quick["margin_gap_vs_avg_last_п.п."] = ((quick[col_mp_last] - avg_last) * 100).round(3)

# Кандидаты: топ-20 по выручке в W-1, среди них самые "ниже средней"
quick_candidates = (
    quick.assign(rev_last=quick[f"Выручка_W{w_last}"])
         .sort_values("rev_last", ascending=False)
         .head(20)
         [["Бренд AX 1", "rev_last", col_mp_last, "margin_gap_vs_avg_last_п.п."]]
         .sort_values("margin_gap_vs_avg_last_п.п.")
         .head(10)
)

from IPython.display import display

# Чтобы эффекты не выглядели -0.00: будем показывать сразу в п.п.
drivers_view = drivers.copy()
drivers_view["ΔВыручка"] = drivers_view["ΔВыручка"].round(0)
drivers_view["ΔМаржа_руб"] = drivers_view["ΔМаржа_руб"].round(0)
drivers_view["ΔМаржинальность_п.п."] = drivers_view["ΔМаржинальность_п.п."].round(3)

# эффекты уже в п.п. (мы их сделали в drivers: effect_*_п.п.)
drivers_view["effect_total_п.п."] = drivers_view["effect_total_п.п."].round(3)
drivers_view["effect_margin_п.п."] = drivers_view["effect_margin_п.п."].round(3)
drivers_view["effect_mix_п.п."] = drivers_view["effect_mix_п.п."].round(3)

cols_out = [
    "Бренд AX 1",
    "ΔВыручка", "ΔМаржа_руб", "ΔМаржинальность_п.п.",
    "effect_total_п.п.", "effect_margin_п.п.", "effect_mix_п.п."
]

print("Топ ухудшения маржинальности (по вкладу в Δ маржинальности, п.п.):")
display(top_neg[cols_out].reset_index(drop=True))

print("Топ улучшения маржинальности (по вкладу в Δ маржинальности, п.п.):")
display(top_pos[cols_out].reset_index(drop=True))

print("Кандидаты на быстрый результат (топ по выручке W-1 и маржа ниже средней):")
qc = quick_candidates.copy()
qc["rev_last"] = qc["rev_last"].round(0)
qc[col_mp_last] = (qc[col_mp_last] * 100).round(2)            # в %
qc["margin_gap_vs_avg_last_п.п."] = qc["margin_gap_vs_avg_last_п.п."].round(2)
display(qc.reset_index(drop=True))



Топ ухудшения маржинальности (по вкладу в Δ маржинальности, п.п.):


,Бренд AX 1,ΔВыручка,ΔМаржа_руб,ΔМаржинальность_п.п.,effect_total_п.п.,effect_margin_п.п.,effect_mix_п.п.
0,Sisley,"-94,733,503.91","-12,335,429.24",1.16,-0.13,0.04,-0.17
1,Clinique,"12,804,388.53","-12,577,013.94",-2.96,-0.12,-0.17,0.05
2,Shiseido,"-85,983,063.20","-12,670,084.90",0.14,-0.07,0.02,-0.09
3,SENSAI,"-35,826,294.60","-5,581,151.77",-0.24,-0.06,-0.00,-0.06
4,Dr Barbara Sturm,"-14,334,985.41","-3,984,442.65",-2.86,-0.04,-0.01,-0.03
5,CHANEL,"-19,383,960.12","-3,979,465.25",-1.05,-0.04,-0.03,-0.01
6,Helena Rubinstein,"-12,382,157.27","-3,374,353.21",-4.94,-0.04,-0.03,-0.01
7,Grown Alchemist,"-15,288,583.50","-3,114,570.60",NaN,-0.04,-0.02,-0.02
8,FILORGA,"-16,214,566.50","-3,171,821.13",0.50,-0.03,0.00,-0.04
9,Dr Brandt,"-20,053,959.99","-2,734,412.43",NaN,-0.03,-0.02,-0.02


Топ улучшения маржинальности (по вкладу в Δ маржинальности, п.п.):


,Бренд AX 1,ΔВыручка,ΔМаржа_руб,ΔМаржинальность_п.п.,effect_total_п.п.,effect_margin_п.п.,effect_mix_п.п.
0,Clarins,"-355,222,424.17","20,727,762.12",5.46,0.30,0.75,-0.46
1,Payot,"102,293,498.19","19,668,556.84",-0.67,0.26,-0.03,0.29
2,Valmont,"75,657,954.08","14,351,054.36",1.03,0.19,0.03,0.16
3,La Mer,"46,010,697.87","8,121,260.23",0.06,0.11,0.00,0.11
4,La Prairie,"10,189,896.97","6,045,028.66",2.13,0.09,0.05,0.03
5,St.Barth,"34,587,298.11","6,431,817.63",-0.27,0.08,-0.00,0.09
6,Kiehls,"20,952,008.69","3,870,925.33",-0.09,0.06,-0.00,0.06
7,Caudalie,"44,411,184.51","3,388,018.50",-2.30,0.05,-0.06,0.11
8,ELEMIS,"15,252,246.99","3,306,528.98",-0.10,0.05,-0.00,0.05
9,OK Beauty,"21,680,162.32","3,678,495.52",0.28,0.05,0.00,0.05


Кандидаты на быстрый результат (топ по выручке W-1 и маржа ниже средней):


,Бренд AX 1,rev_last,Маржинальность_W15,margin_gap_vs_avg_last_п.п.
0,CHANEL,"217,319,320.00",7.73,-7.72
1,Cle De Peau,"126,510,755.00",8.45,-7.00
2,Estee Lauder,"325,930,625.00",10.47,-4.98
3,Christian Dior,"129,024,994.00",10.97,-4.48
4,Biotherm,"217,137,765.00",11.62,-3.84
5,Lancome,"204,288,646.00",12.74,-2.71
6,Clinique,"498,201,173.00",14.01,-1.44
7,SENSAI,"125,798,039.00",14.50,-0.96
8,Clarins,"974,711,948.00",14.59,-0.86
9,BABOR,"290,841,977.00",14.80,-0.66


In [19]:
def fmt_money(x: float) -> str:
    return f"{x:,.0f}".replace(",", " ")

def fmt_pp(x: float) -> str:
    return f"{x:+.2f} п.п."

rev_change_pct = (company_last_rev / company_prev_rev - 1) * 100 if company_prev_rev > 0 else np.nan

note = []
note.append(f"Период сравнения: неделя {w_prev} (W-2) vs {w_last} (W-1).")
note.append(f"Выручка: {fmt_money(company_prev_rev)} → {fmt_money(company_last_rev)} ({rev_change_pct:+.2f}% к W-2).")
note.append(f"Маржинальность: {company_prev_mp*100:.2f}% → {company_last_mp*100:.2f}% ({fmt_pp((company_last_mp-company_prev_mp)*100)}).")
note.append(
    f"Разложение изменения маржинальности: эффект маржи = {fmt_pp(pivot['effect_margin'].sum()*100)}, "
    f"эффект структуры (mix) = {fmt_pp(pivot['effect_mix'].sum()*100)}."
)

note.append("Топ-5 драйверов ухудшения маржинальности (вклад, п.п.):")
for _, r in top_neg.head(5).iterrows():
    note.append(f"- {r['Бренд AX 1']}: {fmt_pp(r['effect_total_п.п.'])} (маржа {fmt_pp(r['effect_margin_п.п.'])}, mix {fmt_pp(r['effect_mix_п.п.'])})")

note.append("Топ-5 драйверов улучшения маржинальности (вклад, п.п.):")
for _, r in top_pos.head(5).iterrows():
    note.append(f"- {r['Бренд AX 1']}: {fmt_pp(r['effect_total_п.п.'])} (маржа {fmt_pp(r['effect_margin_п.п.'])}, mix {fmt_pp(r['effect_mix_п.п.'])})")

note.append("Быстрый результат: бренды с высокой выручкой и маржинальностью ниже средней W-1:")
for _, r in quick_candidates.head(5).iterrows():
    note.append(
        f"- {r['Бренд AX 1']}: выручка W-1 {fmt_money(r['rev_last'])}, "
        f"маржинальность W-1 {r[col_mp_last]*100:.2f}% (отклонение {fmt_pp(r['margin_gap_vs_avg_last_п.п.'])})"
    )

note_text = "\n".join(note)
print(note_text)

# === Экспорт файлов (для репозитория) ===
pivot_view.to_csv("task3_brand_week_compare_view.csv", index=False, encoding="utf-8-sig")
company_summary.to_csv("task3_company_summary.csv", index=False, encoding="utf-8-sig")
effects_summary.to_csv("task3_margin_decomposition.csv", index=False, encoding="utf-8-sig")
drivers.to_csv("task3_brand_drivers.csv", index=False, encoding="utf-8-sig")
quick_candidates.to_csv("task3_quick_candidates.csv", index=False, encoding="utf-8-sig")

with open("task3_note.txt", "w", encoding="utf-8") as f:
    f.write(note_text)

print("Saved: task3_brand_week_compare_view.csv, task3_company_summary.csv, task3_margin_decomposition.csv, "
      "task3_brand_drivers.csv, task3_quick_candidates.csv, task3_note.txt")


Период сравнения: неделя 14 (W-2) vs 15 (W-1).
Выручка: 8 464 188 705 → 8 207 907 632 (-3.03% к W-2).
Маржинальность: 14.63% → 15.45% (+0.82 п.п.).
Разложение изменения маржинальности: эффект маржи = +0.54 п.п., эффект структуры (mix) = +0.28 п.п..
Топ-5 драйверов ухудшения маржинальности (вклад, п.п.):
- Sisley: -0.13 п.п. (маржа +0.04 п.п., mix -0.17 п.п.)
- Clinique: -0.12 п.п. (маржа -0.17 п.п., mix +0.05 п.п.)
- Shiseido: -0.07 п.п. (маржа +0.02 п.п., mix -0.09 п.п.)
- SENSAI: -0.06 п.п. (маржа -0.00 п.п., mix -0.06 п.п.)
- Dr Barbara Sturm: -0.04 п.п. (маржа -0.01 п.п., mix -0.03 п.п.)
Топ-5 драйверов улучшения маржинальности (вклад, п.п.):
- Clarins: +0.30 п.п. (маржа +0.75 п.п., mix -0.46 п.п.)
- Payot: +0.26 п.п. (маржа -0.03 п.п., mix +0.29 п.п.)
- Valmont: +0.19 п.п. (маржа +0.03 п.п., mix +0.16 п.п.)
- La Mer: +0.11 п.п. (маржа +0.00 п.п., mix +0.11 п.п.)
- La Prairie: +0.09 п.п. (маржа +0.05 п.п., mix +0.03 п.п.)
Быстрый результат: бренды с высокой выручкой и маржинальност